# SS3 Part B: STADIO 1 subscriber churn prediction on the client data extract

**Method applied:** the final method chosen in SS2: a gradient-boosted tree classifier (XGBoost), compared against the logistic-regression baseline and the simple "no recent viewing" rule from the SS1 problem statement.
**Task:** predict which STADIO 1 TV subscribers will cancel (voluntarily, by payment failure or by non-renewal) in the next 30 days, early enough for a retention action to change the outcome.

The notebook has five components, each clearly marked with a heading and committed separately:

| # | Component | Marks |
|---|---|---|
| 1 | Dataset loading | 2 |
| 2 | Dataset preprocessing (incl. train / validation / test split) | 5 |
| 3 | Model training (incl. hyper-parameter justification) | 5 |
| 4 | Model validation (unseen test subset) | 5 |
| 5 | Results visualisation (training and unseen) | 3 |

Run top to bottom. Set `STADIO_DATA_DIR` (or place the CSVs in `data/raw/stadioone/`) if your data lives elsewhere.

---
# 1. DATASET LOADING
*(Component 1 of 5)*

In [ ]:
import json, os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
SEED = 42
np.random.seed(SEED)

# Look for the client extract in a few sensible places (first match wins).
CANDIDATES = [os.environ.get("STADIO_DATA_DIR", ""), "data/raw/stadioone", "data/raw", "stadioone",
              "iitaemmanuel_client_data/stadioone", "../data/raw/stadioone"]
DATA_DIR = next((Path(p) for p in CANDIDATES if p and (Path(p) / "table1_subscriptions.csv").exists()), None)
assert DATA_DIR is not None, "Client CSVs not found: set STADIO_DATA_DIR or copy them to data/raw/stadioone/"
OUT_DIR = Path(os.environ.get("STADIO_OUT_DIR", "outputs")); (OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
print("Reading client data from:", DATA_DIR.resolve())

FILES = {"subs": "table1_subscriptions", "pay": "table2_billing_payments", "view": "table3_viewing_sessions",
         "cal": "table4_race_calendar", "app": "table5_app_fantasy_activity", "tix": "table6_ticketing_hospitality",
         "mail": "table7_email_marketing", "social": "table8_social_comments", "prof": "table9a_fan_profile",
         "merch": "table9b_merchandise", "idmap": "table10_identity_mapping"}
# Everything is read as text first: the extract is uncleaned (mixed date formats, casing, stray spaces),
# so typing is done explicitly in the preprocessing component.
raw = {k: pd.read_csv(DATA_DIR / f"{f}.csv", dtype=str, keep_default_na=True) for k, f in FILES.items()}
pd.DataFrame({"table": [FILES[k] for k in raw], "rows": [len(v) for v in raw.values()],
              "columns": [v.shape[1] for v in raw.values()]}).set_index("table")

In [ ]:
# Which tables does the project need? (SS2 data request: churn label from table 1; features from tables 2-7, 9a, 9b, 10; calendar from 4.)
# Table 8 (social posts) is only a 38%-linked *sample*, so absence of posts is not evidence of disengagement: it is left out.
subs0 = raw["subs"].copy()
print("Subscription periods (raw):", len(subs0), "| distinct subscribers:", subs0.subscriber_id.nunique())
print("\nMissing values per column in table 1:\n", subs0.isna().sum()[lambda s: s > 0])
print("\nExact duplicate rows in table 1:", subs0.duplicated().sum())

# Class balance at the subscription-period level, using the churn definition agreed for this project:
# churn = the fan asked to cancel (status Cancelled / Payment failed, i.e. a cancellation_datetime exists).
# Expired periods ended naturally; cancellation_type separates voluntary / payment failure / non-renewal.
s = subs0.drop_duplicates().copy()
s["status_c"] = s.status.str.strip().str.title()
s["churned_period"] = s.cancellation_datetime.notna().astype(int)
print("\nStatus x churn flag:\n", pd.crosstab(s.status_c, s.churned_period))
print("\nChurn proportion over subscription periods: %.1f%%" % (100 * s.churned_period.mean()))
raw["subs"].head()

### What was loaded (dataset summary)

**Source.** 11 CSV extracts from the client (all loaded as text and typed explicitly later). The unit of analysis is built from table 1 (`subscriptions`): **5,837 raw rows = 5,820 distinct subscription periods from 3,800 subscribers** (17 exact duplicates, removed in preprocessing). The other tables are behavioural context: viewing sessions (56,084), payments (6,322), app/fantasy events (43,130), tickets (1,418), emails (29,665), merchandise (2,584), fan profile (3,811), race calendar (61) and the identity map (3,811). Table 8 (social posts) is loaded but not modelled: it is only a sample, and only ~38% of posts can be linked to a subscriber, so "no posts" cannot be read as disengagement.

**Target and classes.** Churn = the fan asked to cancel, had a payment failure, or did not renew (status `Cancelled` / `Payment failed`, i.e. a `cancellation_datetime` exists). Expired periods ended naturally. At subscription-period level this gives **1,340 churned (23.0%) vs 4,497 not churned (77.0%)** (the 58 `Active` periods have not ended).

**Modelling instances.** The prediction task is "will this subscriber cancel in the next 30 days?", scored once per month for every running subscription. After preprocessing this gives **22,958 subscriber-month instances (3,277 subscribers)** with two classes: *stays* (0) and *cancels within 30 days* (1). The **imbalance is about 96.2 : 3.8 (roughly 25 : 1; 871 positives)**, so accuracy is meaningless (always predicting "stays" scores 96%) and the evaluation uses PR-AUC, recall/precision and top-decile lift instead.

**Data quality found in the extract** (the client warned it is uncleaned): mixed casing/whitespace in categories, four different datetime spellings, exact duplicate rows in most tables, negative plan prices, impossible session durations (e.g. -1 s or 7 days) and watch percentages outside 0-100, and 608 low-confidence (<0.9) identity links.